# Lab 7 — Large Language Models (LLMs)

**Student ID:** 2240004223

Completed examples, all supplied tasks, and the optional challenge. The original notebook skips Task 4, so its task numbering is retained.

## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup

Run the cells from top to bottom. The first run requires internet access to install dependencies and download the three public pretrained models. CPU execution is supported; a GPU is optional. Saved model outputs appear below the executed cells.

In [1]:
import importlib.util
import subprocess
import sys
from importlib.metadata import version

packages = {
    "transformers": "transformers>=4.44,<5",
    "sentencepiece": "sentencepiece",
    "torch": "torch"
}
install_packages = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if importlib.util.find_spec("transformers") is not None:
    installed = tuple(int(part) for part in version("transformers").split(".")[:2])
    if not (4, 44) <= installed < (5, 0):
        install_packages.append("transformers>=4.44,<5")
if install_packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_packages], check=True)
print("Required packages are installed.")

Required packages are installed.


In [2]:
import torch
import transformers
from transformers import pipeline, set_seed, AutoTokenizer, AutoModelForSeq2SeqLM

torch.set_num_threads(2)
device = 0 if torch.cuda.is_available() else -1
print("Libraries loaded successfully!")
print("Transformers version:", transformers.__version__)
print("PyTorch version:", torch.__version__)
print("Device:", "GPU" if device == 0 else "CPU")

Libraries loaded successfully!
Transformers version: 4.57.6
PyTorch version: 2.14.1+cpu
Device: CPU


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [3]:
generator = pipeline("text-generation", model="distilgpt2", device=device)
prompt = "Natural language processing is"
output = generator(
    prompt,
    max_new_tokens=30,
    do_sample=False,
    pad_token_id=generator.tokenizer.eos_token_id
)
print(output[0]["generated_text"])

Natural language processing is a common feature of the language processing process.























# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [4]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")
model = model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()
translation_prompt = "Translate to French: I love natural language processing."
translation_inputs = tokenizer(translation_prompt, return_tensors="pt").to(model.device)
with torch.inference_mode():
    translation_output = model.generate(**translation_inputs, max_new_tokens=60, do_sample=False)
translation = tokenizer.decode(translation_output[0], skip_special_tokens=True)
print("English: I love natural language processing.")
print("Model-generated French:", translation)

English: I love natural language processing.
Model-generated French: Je s'agissant de la technologie de linguage naturelle.


### Translation observation

The displayed translation is the model's actual output. FLAN-T5-small gives an inaccurate translation in this run. A correct reference translation is **« J’aime le traitement automatique du langage naturel. »** Small pretrained models can make mistakes even when the code runs correctly.

# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [5]:
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device
)
text = "This NLP lab is easy and interesting."
result = classifier(text)[0]
print("Text:", text)
print("Label:", result["label"])
print(f"Score: {result['score']:.6f}")

Text: This NLP lab is easy and interesting.
Label: POSITIVE
Score: 0.999779


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

In [6]:
model_families = [
    ("Sentiment classification", "Encoder-only"),
    ("Machine translation", "Encoder-decoder"),
    ("Open-ended text generation", "Decoder-only"),
    ("Named Entity Recognition", "Encoder-only"),
    ("Text summarization", "Encoder-decoder")
]
for number, (task, family) in enumerate(model_families, start=1):
    print(f"{number}. {task}: {family}")

1. Sentiment classification: Encoder-only
2. Machine translation: Encoder-decoder
3. Open-ended text generation: Decoder-only
4. Named Entity Recognition: Encoder-only
5. Text summarization: Encoder-decoder


## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [7]:
sentences = [
    "The movie was excellent.",
    "The lecture was difficult to understand.",
    "I enjoyed learning about transformers."
]
sentiment_results = classifier(sentences)
for sentence, prediction in zip(sentences, sentiment_results):
    print("Sentence:", sentence)
    print("Predicted label:", prediction["label"])
    print(f"Score: {prediction['score']:.6f}")
    print()

Sentence: The movie was excellent.
Predicted label: POSITIVE
Score: 0.999861

Sentence: The lecture was difficult to understand.
Predicted label: NEGATIVE
Score: 0.999665

Sentence: I enjoyed learning about transformers.
Predicted label: POSITIVE
Score: 0.999746



## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [8]:
paragraph = (
    "A university opened a new library for its students. "
    "The library has quiet study rooms, computers, and thousands of books. "
    "It stays open until midnight during exams. "
    "Students can also borrow laptops and attend free research workshops."
)
summary_prompt = paragraph + "\nWhat is a one-sentence summary of this paragraph?"
summary_inputs = tokenizer(summary_prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
with torch.inference_mode():
    summary_output = model.generate(
        **summary_inputs,
        max_new_tokens=80,
        num_beams=4,
        do_sample=False,
        no_repeat_ngram_size=3,
        early_stopping=True
    )
summary = tokenizer.decode(summary_output[0], skip_special_tokens=True)
print("Original paragraph:")
print(paragraph)
print("\nGenerated summary:")
print(summary)

Original paragraph:
A university opened a new library for its students. The library has quiet study rooms, computers, and thousands of books. It stays open until midnight during exams. Students can also borrow laptops and attend free research workshops.

Generated summary:
A new library has been opened for its students.



## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**


### Answer

A decoder-only model generates a reply one token at a time using the conversation so far as context. Its causal attention allows each new token to depend on preceding tokens, which suits continuing a dialogue. Because it generates variable-length text, it can produce open-ended answers instead of choosing from fixed labels. Instruction tuning and preference training further help models such as ChatGPT follow requests and respond usefully.

# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.


### Solution

The following loop generates three continuations without resetting the random seed. Sampling is enabled, so the random state advances between calls.

In [9]:
sampling_prompt = "The future of artificial intelligence is"
sampled_outputs = []
for run in range(1, 4):
    sampled_output = generator(
        sampling_prompt,
        max_new_tokens=40,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        top_k=50,
        pad_token_id=generator.tokenizer.eos_token_id
    )[0]["generated_text"]
    sampled_outputs.append(sampled_output)
    print(f"Generation {run}:")
    print(sampled_output)
    print()

Generation 1:
The future of artificial intelligence is very likely, given that it is already being used in the near future. But the biggest obstacle is the speed with which such machines can develop.




The question of the future of

Generation 2:
The future of artificial intelligence is going to require the greatest effort and commitment of our species,” said David H. L. Anderson, a professor of biogeography at the University of Michigan. “This is not a

Generation 3:
The future of artificial intelligence is now more than just a matter of time. Artificial intelligence is a vital part of the human being. It can be combined with technology to make our lives easier and more enjoyable for everyone.






### Why can the outputs differ?

With sampling enabled, the model randomly selects the next token from a probability distribution, so different choices can produce different continuations for the same prompt. Temperature and top-p control the distribution used for sampling. Resetting the same seed before each generation can reproduce the same sampled output under the same execution conditions; setting `do_sample=False` selects tokens greedily. Random sampling can still occasionally produce identical outputs.

DistilGPT-2 is a small text-completion model, so its continuations can be repetitive or inaccurate. Its output should be read as a generation demonstration.

## References

- [FLAN-T5-small model card](https://huggingface.co/google/flan-t5-small)
- [Hugging Face generation documentation](https://huggingface.co/docs/transformers/main_classes/text_generation)